# mcp

> fossick as an MCP server for coding agents

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

The Model Context Protocol (MCP) lets an AI agent, such as Claude Code, Codex, or Cursor, discover and call external tools. This module exposes fossick functions as MCP tools. The agent sees each tool's name, arguments, and docstring. These docstrings tell the agent when to use each option. They define the tool interface and are longer than other fossick docstrings.

Most tools take the arguments of the Python function they wrap. They also take a limit on returned text, such as `max_chars`. A tool cuts a long result at that limit. It adds a note with the number of cut characters. The note tells an agent to ask for more. Tools convert results to plain JSON types before returning them. The server uses standard input and output by default.

In [ ]:
#| export
"""fossick's search, fetch, reader, hidden API and browser tools as an MCP server, started with `fossick-mcp`.

Docs: https://vedicreader.github.io/fossick/mcp.html.md"""

import hmac, json, os, secrets, sys
from pathlib import Path
from fossick.core import (_clone_dir, _gh_remotes, _nb_stem, fetch as _fetch, to_md, crawl as _crawl, fetch_all as _fetch_all,
                          read_arxiv as _read_arxiv, lookup_doi as _lookup_doi,
                          read_yt as _read_yt, search_yt as _search_yt, download_yt as _download_yt,
                          read_gh_repo as _read_gh_repo, read_gh_file as _read_gh_file,
                          find_xhr as _find_xhr, replay_xhr as _replay_xhr,
                          paginate_api as _paginate_api, url2nb as _url2nb, pdf2nb as _pdf2nb)
from fossick.search import search as _search, google as _google, research as _research

try:
    from mcp.server.fastmcp import FastMCP
    from mcp.server.transport_security import TransportSecuritySettings
except ImportError as e:
    raise ImportError("fossick-mcp needs the `mcp` package — reinstall fossick "
                      "with `uv add fossick` or `pip install fossick`") from e

In [ ]:
#| export
_LOOPBACK = ('127.0.0.1', 'localhost', '::1')
_GUARD = TransportSecuritySettings(enable_dns_rebinding_protection=True,
                                   allowed_hosts=['127.0.0.1:*', 'localhost:*', '[::1]:*'],
                                   allowed_origins=['http://127.0.0.1:*', 'http://localhost:*', 'http://[::1]:*'])

mcp = FastMCP('fossick', transport_security=_GUARD, instructions=(
    'Web toolkit. web_search/research for queries; fetch_page for URLs (heavy=JS, stealthy=anti-bot, '
    'session=logged-in Chrome, auto=escalate until not bot-blocked); read_arxiv/read_youtube/read_github_* '
    'for those sites; find_hidden_apis + replay_capture/paginate_api to turn a page into a JSON data feed; '
    'browse + page_* tools drive a persistent logged-in Chrome for interactive/authenticated flows; '
    'shop_* tools drive a shopping cart on any storefront — shop_open, shop_search, shop_add '
    '(verified: it tells you whether the cart actually changed), shop_cart, shop_fields/shop_fill '
    'for checkout forms.'))

#: The largest plain HTTP body a tool takes
MAX_BYTES = 20_000_000

def _on(var:str) -> bool: return (os.environ.get(var) or '').strip().lower() in ('1', 'true', 'yes', 'on')

#: Tools that use the logged-in debug Chrome are on only with FOSSICK_MCP_SESSION=1
SESSION = _on('FOSSICK_MCP_SESSION')
_live = mcp.tool() if SESSION else (lambda f: f)

def _need_session(used=True):
    "Raise `PermissionError` when a call uses the logged-in Chrome and `SESSION` is off."
    if used and not SESSION: raise PermissionError('logged-in Chrome tools are off: set FOSSICK_MCP_SESSION=1')

def _root() -> Path: return Path(os.environ.get('FOSSICK_MCP_ROOT') or os.getcwd()).resolve()

def _inroot(p) -> str:
    "`p` resolved against `_root()`, or `PermissionError` when it is outside."
    r = _root()
    q = (r/Path(p).expanduser()).resolve()
    if not q.is_relative_to(r): raise PermissionError(f'{p} is outside {r}: set FOSSICK_MCP_ROOT to change it')
    return str(q)

def _trunc(s, n):
    'Clip `s` to `n` chars with an explicit truncation marker.'
    s = s or ''
    return s if not n or len(s) <= n else s[:n] + f'\n…[truncated {len(s)-n} of {len(s)} chars]'

def _jsonable(o):
    'Recursively convert result objects (L, Response, Path, ...) to JSON-serializable types.'
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (str, int, float, bool)) or o is None: return o
    if hasattr(o, '__iter__'): return [_jsonable(x) for x in o]
    return str(o)

## Search and research

In [ ]:
#| export
@mcp.tool()
def web_search(query:str, n:int=10, category:str='text', region:str='auto', timelimit:str|None=None,
               google:bool=False) -> list:
    "Search the web with DDGS metasearch. No API key is needed. category accepts text|images|news|videos|books. region accepts a DDGS country-lang code or auto. Set region when the answer depends on location, law, price, or availability. timelimit d|w|m|y keeps recent results. google=True uses Google ranking through a stealth browser."
    res = (_google(query, n=n, region=region, timelimit=timelimit) if google
           else _search(query, category=category, n=n, region=region, timelimit=timelimit))
    return _jsonable(res)

@mcp.tool()
def research(query:str, n:int=5, google:bool=False, region:str='auto', timelimit:str|None=None,
             sel:str|None=None, chars:int=4000) -> dict:
    "Search and read the top n readable results in parallel. Escalate past bot walls, skip bot-wall or empty pages, and backfill skipped results. Return {query, sources, digest, dropped, region}. digest is cited markdown with one ## section per source. Each section contains passages that answer the query and the source publication date. region accepts a DDGS country-lang code or auto. Set it when the answer depends on jurisdiction or market. timelimit d|w|m|y keeps recent sources."
    return _jsonable(_research(query, n=n, engine='google' if google else 'search', region=region,
                               timelimit=timelimit, sel=sel, chars=chars))

@mcp.tool()
def lookup_doi(title:str) -> str:
    "Return the doi.org URL for the first Crossref match on a paper title."
    return _lookup_doi(title) or "not found"

## Fetching and crawling

In [ ]:
#| export
@mcp.tool()
def fetch_page(url:str, sel:str|None=None, heavy:bool=False, stealthy:bool=False, session:bool=False,
               auto:bool=False, max_chars:int=8000) -> dict:
    "Fetch a URL as markdown. Pass sel, a CSS selector, to skip navigation and ads. heavy enables JavaScript rendering. stealthy uses an anti-bot fetcher. session reuses logged-in debug Chrome cookies. auto escalates plain->heavy->stealthy->session. The result gives the selected tier."
    _need_session(session or auto)
    p = _fetch(url, sel=sel, heavy=heavy, stealthy=stealthy, session=session, auto=auto, max_bytes=MAX_BYTES)
    return {'url': p['url'], 'status': p['status'], 'tier': getattr(p, 'tier', None),
            'markdown': _trunc(to_md(p), max_chars)}

@mcp.tool()
def fetch_pages(urls:list, sel:str|None=None, max_chars:int=4000) -> list:
    "Fetch known URLs in parallel. Return [{url, status, markdown}] in the same order as urls."
    return [{'url': p['url'], 'status': p['status'], 'markdown': _trunc(to_md(p, sel=sel), max_chars)}
            for p in _fetch_all(urls, sel=sel, max_bytes=MAX_BYTES)]

@mcp.tool()
def crawl_site(url:str, sel:str|None=None, follow_sel:str='a[href]', max_pages:int=10,
               same_domain:bool=True, heavy:bool=False, stealthy:bool=False, max_chars:int=4000) -> list:
    "Crawl from a start URL by following follow_sel links. Return [{url, status, markdown}] for each page."
    pages = _crawl(url, sel=sel, follow_sel=follow_sel, max_pages=max_pages,
                   same_domain=same_domain, heavy=heavy, stealthy=stealthy,
                   **({} if heavy or stealthy else dict(max_bytes=MAX_BYTES)))
    return [{'url': p['url'], 'status': p['status'], 'markdown': _trunc(to_md(p, sel=sel), max_chars)}
            for p in pages]

## Readers

The module has tools for arXiv papers, YouTube videos and searches, GitHub files and repositories, and page or PDF conversion to notebooks.

In [ ]:
#| export
@mcp.tool()
def read_arxiv(url:str, include_source:bool=False, chars:int=8000, save_dir:str='.') -> dict:
    "Read an arXiv paper from an ID or arXiv URL. Return {title, authors, published, summary, pdf_path}. include_source adds the full text. The full text is 30-100k characters. Increase chars only when needed."
    p = _read_arxiv(url, source=include_source, save_dir=_inroot(save_dir))
    out = {k: _jsonable(v) for k, v in p.items() if k not in ('source','doc')}
    if include_source: out['source'] = _trunc(p.get('source'), chars)
    return out

@mcp.tool()
def read_youtube(url:str, chars:int=20000) -> dict:
    "Read a YouTube video from a URL or ID. Return metadata and the full English transcript in 'source'."
    v = {k: _jsonable(val) for k, val in dict(_read_yt(url)).items()}
    v['source'] = _trunc(v.get('source'), chars)
    return v

@mcp.tool()
def search_youtube(query:str, n:int=10) -> list:
    "Search YouTube. Return [{title, url, channel, duration, description}]."
    return _jsonable(_search_yt(query, n=n))

@mcp.tool()
def download_youtube(url:str, format:str='audio', save_dir:str='.') -> str:
    "Download YouTube audio or video. format accepts audio, video, or any yt-dlp format string. Return the saved file path."
    return str(_download_yt(url, format=format, save_dir=_inroot(save_dir)))

@mcp.tool()
def read_github_file(url:str, max_chars:int=20000) -> str:
    "Read a single file from a GitHub blob URL."
    return _trunc(_read_gh_file(url, max_bytes=MAX_BYTES), max_chars)

@mcp.tool()
def read_github_repo(url:str, globs:list|None=None, limit:int|None=None, max_chars_per_file:int=8000) -> dict:
    "Read files from a GitHub repository URL, SSH address, or local path. Filter files by glob patterns. The default is README*, pyproject.toml, and *.py. Return {path: content}."
    rs = _gh_remotes(url)
    base = (_clone_dir(rs[0]) if rs else Path(url := _inroot(url))).resolve()
    files = _read_gh_repo(url, globs=tuple(globs) if globs else None, limit=limit, as_list=True)
    # a symlink in the tree can point anywhere, so each file must resolve inside the folder
    return {str(p): _trunc(p.read_text(errors='ignore'), max_chars_per_file)
            for p in files if p.resolve().is_relative_to(base)}

@mcp.tool()
def url_to_notebook(url:str, path:str|None=None) -> str:
    "Convert an HTML page, PDF, or arXiv paper URL to a Jupyter notebook. Return the notebook path."
    if not url.startswith(('http://', 'https://')): raise ValueError(f'{url} is not an http or https URL: use pdf_to_notebook for a local PDF')
    return str(_url2nb(url, nb_path=_inroot(path or f'{_nb_stem(url)}.ipynb'), max_bytes=MAX_BYTES))

@mcp.tool()
def pdf_to_notebook(src:str, path:str|None=None, ocr:str='auto') -> str:
    "Convert a PDF URL or local path to a notebook with one markdown cell per page. ocr accepts auto|on|off."
    if not src.startswith('http'): src = _inroot(src)
    return str(_pdf2nb(src, nb_path=_inroot(path or f'{_nb_stem(src)}.ipynb'), ocr_selection=ocr, max_bytes=MAX_BYTES))

## Hidden APIs

`find_hidden_apis` and `capture_network` record and number a page's background requests. The server keeps the list from the most recent call. `replay_capture` sends request `capture_id` from that list as a plain HTTP call with the browser's cookies. `paginate_api` reads every page of a JSON API. It returns at most `max_items` items.

In [ ]:
#| export
_captures = []   # replayable request captures from the last find_hidden_apis/capture_network call

def _stash(caps):
    'Replace the stored captures with `caps`; their indices are the capture_ids.'
    _captures.clear(); _captures.extend(caps)

@mcp.tool()
def find_hidden_apis(url:str, pattern:str='*', session:bool=False, preview_chars:int=500) -> list:
    "Visit a page with a browser and capture the JSON/XHR API calls it makes (glob/regex pattern filters URLs). session=True captures through the logged-in debug Chrome. Each hit has a capture_id for replay_capture."
    _need_session(session)
    hits = _find_xhr(url, pattern=pattern, session=session)
    _stash([h.get('capture') for h in hits])
    return [{'capture_id': i, 'url': h['url'], 'content_type': h.get('content_type'),
             'data_preview': _trunc(json.dumps(_jsonable(h.get('data')), default=str), preview_chars)}
            for i, h in enumerate(hits)]

@mcp.tool()
def replay_capture(capture_id:int, data:str|None=None, max_chars:int=8000) -> dict:
    "Re-issue a captured request (by capture_id from find_hidden_apis/capture_network) as a fast plain-HTTP call, reusing the browser's cookies. data overrides the request body."
    if not (0 <= capture_id < len(_captures)) or _captures[capture_id] is None:
        raise ValueError(f'no capture #{capture_id} — run find_hidden_apis or capture_network first')
    r = _replay_xhr(_captures[capture_id], data=data, use_cookies=SESSION, max_bytes=MAX_BYTES)
    try: body = r.json()
    except Exception: body = getattr(r, 'text', None) or getattr(r, 'body', None) or str(r)
    if not isinstance(body, str): body = json.dumps(_jsonable(body), default=str)
    return {'status': getattr(r, 'status', None), 'body': _trunc(body, max_chars)}

@mcp.tool()
def paginate_api(url:str, payload:dict|None=None, page_field:str='pageNumber', size_field:str='pageSize',
                 results_field:str|None=None, method:str='POST', max_pages:int=10, page_start:int=1,
                 max_items:int=500) -> list:
    "Paginate a JSON API, collecting all items across pages. payload is the base body (POST) or params (GET); page_field is the key incremented per page; results_field is auto-detected if None."
    items = _paginate_api(url, payload=payload, page_field=page_field, size_field=size_field,
                          results_field=results_field, method=method, max_pages=max_pages, page_start=page_start,
                          max_bytes=MAX_BYTES)
    return _jsonable(items[:max_items])

## Browser and shop

`browse` opens a page in debug Chrome and returns its snapshot. The server remembers that page. The `page_*` tools act on it until the next `browse`. The `shop_*` tools use one shopping session opened by `shop_open`. They wrap `fossick.shop`.

In [ ]:
#| export
_browser = {'page': None}   # current debug-Chrome page driven by browse/page_* tools

def _page():
    'The page opened by the last `browse` call.'
    if _browser['page'] is None: raise ValueError('no open page — call browse(url) first')
    return _browser['page']

@_live
def browse(url:str, port:int=9223, full:bool=False) -> str:
    "Open a URL in persistent debug Chrome and return a compact accessibility snapshot. Each element uses '[#id] role \"name\"'. Later page_* tools act on this page. full=True includes non-interactive elements."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect(port=port)
        return await cdp.open_page(url)
    _browser['page'] = pg = syncy(_run())
    return syncy(pg.snapshot(interactive=not full))

@_live
def page_snapshot(full:bool=False) -> str:
    "Re-read the current page's accessibility snapshot (after JS/DOM changes)."
    from fossick.cdp import syncy
    return syncy(_page().snapshot(interactive=not full))

@_live
def page_fill_form(fields:dict, submit:str|None=None) -> str:
    "Fill current-page form fields by visible label with {label: value}. This handles <select>. Optionally click a submit button by label. Return the post-action snapshot."
    from fossick.cdp import syncy
    return syncy(_page().fill_form(fields, submit=submit))

@_live
def page_act(steps:list) -> dict:
    "Run a declarative flow on the current page. Use these JSON array steps: ['goto',url], ['fill',label,value], ['click',label], ['select',label,option], ['wait',text], ['wait_sel',css], ['read',css], or ['read',css,label]. Return {label: markdown} for every read step."
    from fossick.cdp import syncy
    return _jsonable(syncy(_page().act([tuple(s) for s in steps])))

@_live
def page_markdown(sel:str|None=None, max_chars:int=8000) -> str:
    "Read the current page's live post-JavaScript DOM as markdown. Optionally narrow it with a CSS selector."
    from fossick.cdp import syncy
    return _trunc(syncy(_page().md(sel=sel)), max_chars)

@_live
def capture_network(url:str, pattern:str='.*', tail:int=3, port:int=9223, preview_chars:int=300) -> list:
    "Navigate debug Chrome to a URL and capture outgoing requests matching pattern. Listen for tail seconds after load. Give each request a capture_id for replay_capture."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect(port=port)
        try: return await cdp.calls(url=url, pattern=pattern, tail=tail)
        finally: await cdp.close()
    vals = list(syncy(_run()).values())
    _stash(vals)
    return [{'capture_id': i, 'url': r['url'], 'method': r.get('method'),
             'response_preview': _trunc(str(r.get('response_body') or ''), preview_chars)}
            for i, r in enumerate(vals)]

In [ ]:
#| export
_shop_session = {}   # the one shopping session every shop_* tool acts on

def _sh(url=None, port=9223):
    "Return the one shopping session. Open it at `url`, or move it there, when `url` is given."
    from fossick.shop import shop as _open
    if (s := _shop_session.get('s')) is None:
        if url is None: raise ValueError('no shop open — call shop_open(url) first')
        _shop_session['s'] = s = _open(url, port=port)
    elif url: s.open(url)
    return s

@_live
def shop_open(url:str, port:int=9223) -> dict:
    "Open a store in persistent debug Chrome or move the session to `url`. Use a category or product page. The site root keeps the current page. Report the platform, blockers such as cookie banners or login, the current cart, and page products. Later shop_* tools act on this session."
    s = _sh(url, port=port)
    return {'url': s.url, 'platform': s.platform(), 'blockers': s.blockers(),
            'cart': _jsonable(s.cart()), 'products': _jsonable(s.products(25))}

@_live
def shop_search(query:str, limit:int=25) -> list:
    "Search the open store and return numbered products [{i, title, price, url, add, qty, oos}]. Pass i or the title to shop_add."
    return _jsonable(_sh().search(query, limit=limit))

@_live
def shop_products(limit:int=25) -> list:
    "Products on the page the session is currently on, numbered for shop_add."
    return _jsonable(_sh().products(limit))

@_live
def shop_add(item:str, qty:int=1, variant:str|None=None) -> dict:
    "Add a product to the cart by index or title, then verify the change. ok=True means the cart changed. ok=None means the site provides no signal to check. need='variant' means choose one returned variant."
    return _jsonable(_sh().add(item, qty=qty, variant=variant))

@_live
def shop_cart(page:bool=False) -> dict:
    "Return the current cart. page=True opens the cart page first. This provides the line indexes for shop_line."
    s = _sh()
    return _jsonable(s.cart_page() if page else s.cart(lines=True))

@_live
def shop_line(line:int, qty:int) -> dict:
    "Set a cart line from shop_cart(page=True) to `qty`, or set it to 0 to remove it. Verify the change."
    s = _sh()
    return _jsonable(s.remove(line) if qty == 0 else s.set_qty(line, qty))

@_live
def shop_dismiss() -> dict:
    "Click through cookie/consent banners and closable modals blocking the store."
    s = _sh()
    return {'clicked': s.dismiss(), 'blockers': s.blockers()}

@_live
def shop_fields() -> list:
    "Return every visible form field with its label, autocomplete token, current value, and select options. Use this output to fill a checkout without guessing field names."
    return _jsonable(_sh().fields())

@_live
def shop_fill(profile:dict, submit:str|None=None, confirm:bool=False) -> dict:
    "Fill a checkout form from a profile dict with first_name, last_name, email, phone, address1, address2, city, state, postcode, country, company, notes, and card_* fields. Match fields by autocomplete token first. Re-read the form to confirm the values. A numeric key sets the shop_fields index directly for options without a profile name, such as size, colour, or delivery window. submit= clicks that button. A payment button also needs confirm=True."
    if confirm and not _on('FOSSICK_MCP_PAY'): raise PermissionError('a payment needs a person: set FOSSICK_MCP_PAY=1')
    return _jsonable(_sh().fill(profile, submit=submit, confirm=confirm))

In [ ]:
#| export
def _bearer(app, token:str):
    "Put `app` behind an `Authorization: Bearer <token>` check for each HTTP request."
    want = f'Bearer {token}'.encode()
    async def guard(scope, receive, send):
        if scope['type'] == 'http' and not hmac.compare_digest(dict(scope['headers']).get(b'authorization', b''), want):
            await send({'type': 'http.response.start', 'status': 401, 'headers': [(b'www-authenticate', b'Bearer')]})
            return await send({'type': 'http.response.body', 'body': b'unauthorized'})
        await app(scope, receive, send)
    return guard

def _serve_http():
    "Serve Streamable HTTP on loopback only. Use the token in FOSSICK_MCP_TOKEN, or create one and print it to stderr."
    import uvicorn
    if (h := mcp.settings.host) not in _LOOPBACK: raise SystemExit(f'fossick-mcp --http binds to loopback only, not {h}')
    if not (tok := os.environ.get('FOSSICK_MCP_TOKEN')):
        tok = secrets.token_urlsafe(32)
        print(f'fossick-mcp: FOSSICK_MCP_TOKEN={tok}', file=sys.stderr)
    uvicorn.run(_bearer(mcp.streamable_http_app(), tok), host=h, port=mcp.settings.port)

def main():
    "Run the MCP server over standard input and output, or use Streamable HTTP with `--http`."
    _serve_http() if '--http' in sys.argv[1:] else mcp.run(transport='stdio')

## Connecting a client

```sh
uvx fossick-mcp          # standard input and output
fossick-mcp --http       # Streamable HTTP
```

Configure your agent to run the first command as an MCP server. If your project already depends on fossick, run `uv run fossick-mcp`. This uses the project environment and installs no separate copy.

## Safety settings

A page that the agent reads can contain instructions for the agent. The server is strict by default. Use these environment variables to change its settings.

| Variable | Default | Effect |
|---|---|---|
| `FOSSICK_MCP_SESSION=1` | off | Turns on the tools that use the logged-in debug Chrome: `browse`, `page_*`, `capture_network`, `shop_*`, `session=`, `auto=` and the cookies of `replay_capture`. |
| `FOSSICK_MCP_PAY=1` | off | Lets `shop_fill(confirm=True)` click a payment button. |
| `FOSSICK_MCP_ROOT` | working folder | The one folder that tools can read local files from and write files to. |
| `FOSSICK_MCP_TOKEN` | new each start | The bearer token for `--http`. With none set, the server makes one and prints it to stderr. |

`--http` binds only to a loopback address. It refuses requests with an incorrect `Host` or `Origin` header. Plain HTTP tools stop at `MAX_BYTES` (20 MB) per page.

In [ ]:
#| hide
from fastcore.test import test_eq
from pathlib import Path
import inspect, fossick.mcp as m

# module callables + FastMCP registry (names the client sees)
_need = ('web_search','research','lookup_doi','fetch_page','fetch_pages','crawl_site',
         'read_arxiv','read_youtube','search_youtube','read_github_file','read_github_repo',
         'find_hidden_apis','capture_network','replay_capture','browse','page_snapshot','page_markdown',
         'page_act','page_fill_form','paginate_api','download_youtube',
         'shop_open','shop_search','shop_products','shop_add','shop_cart','shop_line','shop_dismiss',
         'shop_fields','shop_fill')
for name in _need:
    assert hasattr(m, name) and callable(getattr(m, name)), name
_reg = getattr(getattr(m.mcp, '_tool_manager', None), '_tools', None) or getattr(m.mcp, '_tools', {})
_live = {'browse','page_snapshot','page_markdown','page_act','page_fill_form','capture_network',
         'shop_open','shop_search','shop_products','shop_add','shop_cart','shop_line','shop_dismiss',
         'shop_fields','shop_fill'}
if _reg:
    for name in _need:
        assert (name in _reg) == (m.SESSION or name not in _live), f'mcp registry is wrong for {name}'

# hardening: session gate, payment gate, root, bearer token
from fastcore.test import test_fail
import os, tempfile
from fossick.core import syncy
if not m.SESSION:
    test_fail(lambda: m.fetch_page('https://example.com', session=True), contains='FOSSICK_MCP_SESSION')
    test_fail(lambda: m.fetch_page('https://example.com', auto=True), contains='FOSSICK_MCP_SESSION')
    test_fail(lambda: m.find_hidden_apis('https://example.com', session=True), contains='FOSSICK_MCP_SESSION')
if not m._on('FOSSICK_MCP_PAY'): test_fail(lambda: m.shop_fill({}, confirm=True), contains='FOSSICK_MCP_PAY')
with tempfile.TemporaryDirectory() as d:
    os.environ['FOSSICK_MCP_ROOT'] = d
    try:
        r = str(Path(d).resolve())
        test_eq(m._inroot('a/b.ipynb'), f'{r}/a/b.ipynb')
        for bad in ('../x', '/etc/passwd', '~/.ssh'): test_fail(lambda: m._inroot(bad), contains='outside')
        test_fail(lambda: m.read_github_repo('/etc'), contains='outside')
        test_fail(lambda: m.url_to_notebook('https://example.com', path='/tmp/x.ipynb'), contains='outside')
    finally: del os.environ['FOSSICK_MCP_ROOT']

# a symlink inside the root does not let a repository read leave it; a local file is not a URL
with tempfile.TemporaryDirectory() as d:
    root = Path(d)/'root'; root.mkdir()
    (Path(d)/'outside.txt').write_text('outside')
    (root/'inside.txt').write_text('inside')
    (root/'linked.txt').symlink_to(Path(d)/'outside.txt')
    os.environ['FOSSICK_MCP_ROOT'] = str(root)
    try:
        test_eq(sorted(Path(k).name for k in m.read_github_repo('.', globs=['*.txt'])), ['inside.txt'])
        test_fail(lambda: m.url_to_notebook(str(Path(d)/'outside.pdf')), contains='http')
    finally: del os.environ['FOSSICK_MCP_ROOT']

async def _app(scope, receive, send): await send({'type': 'http.response.start', 'status': 200, 'headers': []})
def _status(hdrs):
    out = []
    async def send(msg): out.append(msg)
    syncy(m._bearer(_app, 'tok')({'type': 'http', 'headers': hdrs}, None, send))
    return out[0]['status']
test_eq(_status([]), 401)
test_eq(_status([(b'authorization', b'Bearer nope')]), 401)
test_eq(_status([(b'authorization', b'Bearer tok')]), 200)

# response shaping — full feature block
test_eq(m._trunc(None, 5), '')
test_eq(m._trunc('abcd', 0), 'abcd')
test_eq(m._trunc('abcdefghij', 5)[:5], 'abcde')
assert 'truncated' in m._trunc('abcdefghij', 5)
test_eq(m._jsonable(None), None)
test_eq(m._jsonable([1,'x',True]), [1,'x',True])
test_eq(m._jsonable({'p': Path('.')}), {'p': '.'})
test_eq(m._jsonable({'a':1, 'b':[2, Path('/tmp')]}), {'a':1, 'b':[2, '/tmp']})

# agent-facing signatures
for fn, params in (
    (m.research, ('query','n','region','timelimit')),
    (m.web_search, ('query','n','region','timelimit','google')),
    (m.fetch_page, ('url','sel','heavy','stealthy','session','auto','max_chars')),
    (m.shop_fill, ('profile','submit','confirm')),
):
    ps = set(inspect.signature(fn).parameters)
    assert set(params) <= ps, (fn.__name__, set(params)-ps)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()